In [1]:
today_date = '2025-08-25'

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 3, Finished, Available, Finished)

In [2]:
from pyspark.sql.functions import col

df = spark.read.format('delta').load('abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Tables/dbo/tblsales_silver').filter(col('processing_date')==str(today_date))

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 4, Finished, Available, Finished)

In [3]:
display(df)

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 0a1a8b1c-1090-42f5-b361-3296a9569a68)

In [5]:
df.printSchema()

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 7, Finished, Available, Finished)

root
 |-- Row_ID: string (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Postal_Code: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Market: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Shipping_Cost: double (nullable = true)
 |-- Order_Priority: string (nullable = true)
 |-- Month: string (nu

### Creating Dimension Tables

#### Dim_customer Tables

In [7]:
from pyspark.sql.types import StructType , StructField , StringType,IntegerType,DoubleType,DateType

from delta.tables import DeltaTable


#Define schemas from dimension tables

dim_customer_schema = StructType([
    StructField('Customer_ID', StringType(),True ),
    StructField('Customer_Name',StringType(),True),
    StructField('Segment',StringType(),True),
    StructField('Postal_Code',StringType(),True),
    StructField('City',StringType(),True),
    StructField('State',StringType(),True),
    StructField('Country',StringType(),True),
    StructField('Region',StringType(),True),
    StructField('Market',StringType(),True)
])


DeltaTable.createIfNotExists(spark).tableName('Dim_Customer').addColumns(dim_customer_schema).execute()

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 9, Finished, Available, Finished)

### Dim Product 

In [8]:
dim_product_schema = StructType([
    StructField('Product_ID',StringType(),True),
    StructField('Category',StringType(),True),
    StructField('Sub_Category',StringType(),True),
    StructField('Product_Name',StringType(),True)
])

DeltaTable.createIfNotExists(spark).tableName('Dim_Product').addColumns(dim_product_schema).execute()

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 10, Finished, Available, Finished)

### Fact Sales tables

In [9]:
Fact_Sales_schema = StructType([
    StructField('Row_ID',StringType(),True),
    StructField('Order_ID',StringType(),True),
    StructField('Customer_ID',StringType(),True),
    StructField('Product_ID',StringType(),True),
    StructField('Order_Date',DateType(),True),
    StructField('Ship_Date',DateType(),True),
    StructField('Ship_Mode',StringType(),True),
    StructField('Sales',DoubleType(),True),
    StructField('Quantity',IntegerType(),True),
    StructField('Discount',DoubleType(),True),
    StructField('Profit',DoubleType(),True),
    StructField('Shipping_Cost',DoubleType(),True),
    StructField('Order_Priority',StringType(),True),
    StructField('Month',StringType(),True),
    StructField('Year',StringType(),True),
    StructField('processing_date',StringType(),True),
    StructField('Delivery_Days',StringType(),True),
    StructField('Profit_Margin',StringType(),True)
])


DeltaTable.createIfNotExists(spark).tableName('Fact_Sales').addColumns(Fact_Sales_schema).execute()

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 11, Finished, Available, Finished)

In [10]:
### Loading Data into Dim_Customer 

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 12, Finished, Available, Finished)

In [11]:
df_selected_dim_customer = (
    df.select('Customer_ID',
			'Customer_Name',
			'Segment',
			'Postal_Code',
			'City',
			'State',
			'Country',
			'Region',
			'Market').dropDuplicates())

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 13, Finished, Available, Finished)

In [12]:
from delta.tables import *

dim_customer_table_path = 'abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Tables/dbo/dim_customer'
dim_deltacustomer = DeltaTable.forPath(spark,dim_customer_table_path)

## Perform the MERGE (UPSERT)

dim_deltacustomer.alias('target').merge(
df_selected_dim_customer.alias('source'),'target.Customer_ID=source.Customer_ID'
).whenMatchedUpdate( set = {
			'Customer_Name':'source.Customer_Name',
			'Segment':'source.Segment',
			'Postal_Code':'source.Postal_Code',
			'City':'source.City',
			'State':'source.State',
			'Country':'source.Country',
			'Region':'source.Region',
			'Market':'source.Market'

}).whenNotMatchedInsert(values={
            'Customer_ID':'source.Customer_ID',
			'Customer_Name':'source.Customer_Name',
			'Segment':'source.Segment',
			'Postal_Code':'source.Postal_Code',
			'City':'source.City',
			'State':'source.State',
			'Country':'source.Country',
			'Region':'source.Region',
			'Market':'source.Market'

}).execute()



StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 14, Finished, Available, Finished)

In [13]:
history_df = dim_deltacustomer.history(1)  
operation_metrics = history_df.select("operationMetrics").collect()[0][0]
rows_inserted = operation_metrics.get('numTargetRowsInserted', 0)
rows_updated = operation_metrics.get('numTargetRowsUpdated', 0)
rows_deleted = operation_metrics.get('numTargetRowsDeleted', 0)
rows_affected = int(rows_inserted) + int(rows_updated) + int(rows_deleted) 

print('Total rows of table: ',dim_deltacustomer.toDF().count())
print(f"Rows inserted: {rows_inserted}")
print(f"Rows updated: {rows_updated}")
print(f"Rows deleted: {rows_deleted}")
print(f"Total rows affected: {rows_affected}")

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 15, Finished, Available, Finished)

Total rows of table:  849
Rows inserted: 849
Rows updated: 0
Rows deleted: 0
Total rows affected: 849


### Loading Data into dim_Product 

In [14]:
df_selected_dim_product = (df.select('Product_ID',
			'Category',
			'Sub_Category',
			'Product_Name').dropDuplicates()
            )

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 16, Finished, Available, Finished)

In [15]:
from delta.tables import *

dim_product_table_path = 'abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Tables/dbo/dim_product'
dim_deltaproduct = DeltaTable.forPath(spark,dim_product_table_path)

dim_deltaproduct.alias('target').merge(
    df_selected_dim_product.alias('source'),'target.Product_ID=source.Product_ID'

).whenMatchedUpdate(set = {
			'Category':'source.Category',
			'Sub_Category':'source.Sub_Category',
			'Product_Name':'source.Product_Name'

}).whenNotMatchedInsert(values={
            'Product_ID':'source.Product_ID',
			'Category':'source.Category',
			'Sub_Category':'source.Sub_Category',
			'Product_Name':'source.Product_Name'
}).execute()

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 17, Finished, Available, Finished)

In [16]:
history_df = dim_deltaproduct.history(1)  
operation_metrics = history_df.select("operationMetrics").collect()[0][0]
rows_inserted = operation_metrics.get('numTargetRowsInserted', 0)
rows_updated = operation_metrics.get('numTargetRowsUpdated', 0)
rows_deleted = operation_metrics.get('numTargetRowsDeleted', 0)
rows_affected = int(rows_inserted) + int(rows_updated) + int(rows_deleted) 

print('Total rows of table: ',dim_deltaproduct.toDF().count())
print(f"Rows inserted: {rows_inserted}")
print(f"Rows updated: {rows_updated}")
print(f"Rows deleted: {rows_deleted}")
print(f"Total rows affected: {rows_affected}")

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 18, Finished, Available, Finished)

Total rows of table:  1185
Rows inserted: 1185
Rows updated: 0
Rows deleted: 0
Total rows affected: 1185


### Loading into Fact Sales

In [17]:
df_selected_fact_sales = (df.select(
            'Row_ID',
			'Order_ID',
		    'Customer_ID',
			'Product_ID',
			'Order_Date',
			'Ship_Date',
			'Ship_Mode',
			'Sales',
			'Quantity',
			'Discount',
			'Profit',
			'Shipping_Cost',
			'Order_Priority',
			'Month',
			'Year',
			'processing_date',
			'Delivery_Days',
			'Profit_Margin'
            ).dropDuplicates()
            )

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 19, Finished, Available, Finished)

In [18]:
from delta.tables import *

fact_sales_table_path = 'abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Tables/dbo/fact_sales'

fact_deltasales = DeltaTable.forPath(spark,fact_sales_table_path)

fact_deltasales.alias('target').merge(
    df_selected_fact_sales.alias('source'),'target.Order_ID=source.Order_ID'

).whenMatchedUpdate(set = {
		    'Row_ID':'source.Row_ID',
			# 'Order_ID':'source.Order_ID',
		    'Customer_ID':'source.Customer_ID',
			'Product_ID':'source.Product_ID',
			'Order_Date':'source.Order_Date',
			'Ship_Date':'source.Ship_Date',
			'Ship_Mode':'source.Ship_Mode',
			'Sales':'source.Sales',
			'Quantity':'source.Quantity',
			'Discount':'source.Discount',
			'Profit':'source.Profit',
			'Shipping_Cost':'source.Shipping_Cost',
			'Order_Priority':'source.Order_Priority',
			'Month':'source.Month',
			'Year':'source.Year',
			'processing_date':'source.processing_date',
			'Delivery_Days':'source.Delivery_Days',
			'Profit_Margin':'source.Profit_Margin'
}).whenNotMatchedInsert(values={
            'Row_ID':'source.Row_ID',
			'Order_ID':'source.Order_ID',
		    'Customer_ID':'source.Customer_ID',
			'Product_ID':'source.Product_ID',
			'Order_Date':'source.Order_Date',
			'Ship_Date':'source.Ship_Date',
			'Ship_Mode':'source.Ship_Mode',
			'Sales':'source.Sales',
			'Quantity':'source.Quantity',
			'Discount':'source.Discount',
			'Profit':'source.Profit',
			'Shipping_Cost':'source.Shipping_Cost',
			'Order_Priority':'source.Order_Priority',
			'Month':'source.Month',
			'Year':'source.Year',
			'processing_date':'source.processing_date',
			'Delivery_Days':'source.Delivery_Days',
			'Profit_Margin':'source.Profit_Margin'
}).execute()

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 20, Finished, Available, Finished)

In [19]:
history_df = fact_deltasales.history(1)  
operation_metrics = history_df.select("operationMetrics").collect()[0][0]
rows_inserted = operation_metrics.get('numTargetRowsInserted', 0)
rows_updated = operation_metrics.get('numTargetRowsUpdated', 0)
rows_deleted = operation_metrics.get('numTargetRowsDeleted', 0)
rows_affected = int(rows_inserted) + int(rows_updated) + int(rows_deleted) 

print('Total rows of table: ',fact_deltasales.toDF().count())
print(f"Rows inserted: {rows_inserted}")
print(f"Rows updated: {rows_updated}")
print(f"Rows deleted: {rows_deleted}")
print(f"Total rows affected: {rows_affected}")

StatementMeta(, f5733e36-c1e9-4288-be7e-caf6843e9b2c, 21, Finished, Available, Finished)

Total rows of table:  1675
Rows inserted: 1675
Rows updated: 0
Rows deleted: 0
Total rows affected: 1675
